<a href="https://colab.research.google.com/github/l2Aquel/Pyspark-Data-Analysis/blob/main/PysparkProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("PysparkProjectFile").getOrCreate()

In [ ]:
df = spark.read.csv('Employees.csv',inferSchema=True,header=True)
df.show(truncate=False)

+---+----------+-----------+------+----------+----------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+
|Id |First Name|Last Name  |Gender|Start Date|Department            |Country             |Center|Monthly Salary|Annual Salary|Sick Leaves|Unpaid Leaves|Overtime Hours|
+---+----------+-----------+------+----------+----------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+
|1  |Ghadir    |Hmshw      |Male  |04-04-2018|Quality Control       |Egypt               |West  |1560          |18720        |1          |0            |183           |
|2  |Omar      |Hishan     |Male  |21-05-2020|Quality Control       |Saudi Arabia        |West  |3247          |38964        |0          |5            |198           |
|3  |Ailya     |Sharaf     |Female|28-09-2017|Major Mfg Projects    |Saudi Arabia        |West  |2506          |30072        |0          |3            |192     

Q1. Create a new column called Full Name by combining First Name and Last Name separated by a space. Then, drop the original first and last name columns.

In [ ]:
from pyspark.sql.functions import concat_ws

df.withColumn('Full Name',concat_ws(' ',df['First Name'],df['Last Name'])).drop('First Name','Last Name').show(truncate=False)

+---+------+----------+----------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+------------------+
|Id |Gender|Start Date|Department            |Country             |Center|Monthly Salary|Annual Salary|Sick Leaves|Unpaid Leaves|Overtime Hours|Full Name         |
+---+------+----------+----------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+------------------+
|1  |Male  |04-04-2018|Quality Control       |Egypt               |West  |1560          |18720        |1          |0            |183           |Ghadir Hmshw      |
|2  |Male  |21-05-2020|Quality Control       |Saudi Arabia        |West  |3247          |38964        |0          |5            |198           |Omar Hishan       |
|3  |Female|28-09-2017|Major Mfg Projects    |Saudi Arabia        |West  |2506          |30072        |0          |3            |192           |Ailya Sharaf      |
|4  |Male  |14-0

Q2. Format the schema to use snake_case by replacing spaces with underscores in any column names that contain them.

In [ ]:
df.withColumnRenamed('First Name','First_Name') \
.withColumnRenamed('Last Name','Last_Name') \
.withColumnRenamed('Start Date','Start_Date') \
.withColumnRenamed('Monthly Salary','Monthly_Salary') \
.withColumnRenamed('Annual Salary','Annual_Salary') \
.withColumnRenamed('Sick Leaves','Sick_Leaves') \
.withColumnRenamed('Unpaid Leaves','Unpaid_Leaves') \
.withColumnRenamed('Overtime Hours','Overtime_Hours') \
.show(truncate=False)

+---+----------+-----------+------+----------+----------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+
|Id |First_Name|Last_Name  |Gender|Start_Date|Department            |Country             |Center|Monthly_Salary|Annual_Salary|Sick_Leaves|Unpaid_Leaves|Overtime_Hours|
+---+----------+-----------+------+----------+----------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+
|1  |Ghadir    |Hmshw      |Male  |04-04-2018|Quality Control       |Egypt               |West  |1560          |18720        |1          |0            |183           |
|2  |Omar      |Hishan     |Male  |21-05-2020|Quality Control       |Saudi Arabia        |West  |3247          |38964        |0          |5            |198           |
|3  |Ailya     |Sharaf     |Female|28-09-2017|Major Mfg Projects    |Saudi Arabia        |West  |2506          |30072        |0          |3            |192     

Q3. Extract and display a list containing only the unique departments present in the dataset

In [ ]:
df.select('Department').distinct().orderBy('Department').show(truncate=False)

+---------------------------+
|Department                 |
+---------------------------+
|Account Management         |
|Creative                   |
|Environmental Compliance   |
|Environmental Health/Safety|
|Facilities/Engineering     |
|Green Building             |
|Human Resources            |
|IT                         |
|Major Mfg Projects         |
|Manufacturing              |
|Manufacturing Admin        |
|Marketing                  |
|Product Development        |
|Professional Training Group|
|Quality Assurance          |
|Quality Control            |
|Research Center            |
|Research/Development       |
|Sales                      |
|Training                   |
+---------------------------+



Q4. Find the total number of employees and the average Monthly Salary for each Department. Rename the aggregated columns to Total_Employees and Avg_Monthly_Salary.

In [ ]:
from pyspark.sql.functions import col,count,avg,round

df.groupBy('Department').agg(
    count('Department').alias('Total_Employees'),
    round(avg('Monthly Salary'),2).alias('Avg_Monthly_Salary')
).show(truncate=False)

+---------------------------+---------------+------------------+
|Department                 |Total_Employees|Avg_Monthly_Salary|
+---------------------------+---------------+------------------+
|Facilities/Engineering     |58             |2285.28           |
|Environmental Health/Safety|9              |2000.44           |
|Product Development        |34             |1964.71           |
|Sales                      |20             |1956.45           |
|Professional Training Group|14             |2040.43           |
|Quality Control            |89             |2053.96           |
|Major Mfg Projects         |8              |2242.38           |
|Quality Assurance          |67             |2083.93           |
|Training                   |16             |2361.75           |
|Environmental Compliance   |13             |2508.15           |
|Marketing                  |48             |2061.13           |
|IT                         |40             |2114.53           |
|Research/Development    

Q5. For each Country and Center combination, find the maximum Annual Salary, total Sick Leaves, and average Overtime Hours.

In [ ]:
from pyspark.sql.functions import max,sum,avg,round

df.groupBy('Country','Center').agg(
    max('Annual Salary').alias('Max_Annual_Salary'),
    sum('Sick Leaves').alias('Total_Sick_Leaves'),
    round(avg('Overtime Hours'),2).alias('Avg_Overtime_Hours')
).orderBy('Country','Center').show()

+--------------------+------+-----------------+-----------------+------------------+
|             Country|Center|Max_Annual_Salary|Total_Sick_Leaves|Avg_Overtime_Hours|
+--------------------+------+-----------------+-----------------+------------------+
|               Egypt|  East|            41016|               54|              4.71|
|               Egypt|  Main|            41136|              195|             18.62|
|               Egypt| North|            41400|              201|             13.86|
|               Egypt| South|            40920|               45|             16.58|
|               Egypt|  West|            41352|              120|             13.08|
|             Lebanon|  Main|            39420|                8|              17.2|
|             Lebanon| North|            40044|                2|              5.33|
|             Lebanon| South|            24816|                0|               0.5|
|             Lebanon|  West|            11964|                1|

Q6. Pivot the data to display the total Monthly Salary paid out for each Department, broken down by Gender

In [ ]:
from pyspark.sql.functions import sum

df.groupBy('Department').pivot('Gender').sum('Monthly Salary').show(truncate=False)

+---------------------------+------+------+
|Department                 |Female|Male  |
+---------------------------+------+------+
|Facilities/Engineering     |51973 |80573 |
|Environmental Health/Safety|5705  |12299 |
|Sales                      |15141 |23988 |
|Product Development        |31911 |34889 |
|Professional Training Group|5240  |23326 |
|Quality Control            |53411 |129391|
|Major Mfg Projects         |8333  |9606  |
|Quality Assurance          |60418 |79205 |
|Training                   |8121  |29667 |
|Environmental Compliance   |8125  |24481 |
|Marketing                  |28171 |70763 |
|IT                         |34006 |50575 |
|Research/Development       |3963  |6530  |
|Green Building             |8088  |8575  |
|Manufacturing Admin        |NULL  |9605  |
|Creative                   |14193 |24022 |
|Manufacturing              |84491 |196158|
|Account Management         |66140 |96582 |
|Research Center            |1230  |8205  |
|Human Resources            |551

Q7. Find the employee with the second highest Monthly Salary in the entire company

In [ ]:
from pyspark.sql.functions import desc

df.orderBy(desc('Monthly Salary')).limit(2) \
.orderBy('Monthly Salary').limit(1).show()

+---+----------+---------+------+----------+------------------+-------+------+--------------+-------------+-----------+-------------+--------------+
| Id|First Name|Last Name|Gender|Start Date|        Department|Country|Center|Monthly Salary|Annual Salary|Sick Leaves|Unpaid Leaves|Overtime Hours|
+---+----------+---------+------+----------+------------------+-------+------+--------------+-------------+-----------+-------------+--------------+
|348|     Dalia|   Hamdan|Female|11-02-2019|Major Mfg Projects|  Egypt|  West|          3446|        41352|          0|            0|            10|
+---+----------+---------+------+----------+------------------+-------+------+--------------+-------------+-----------+-------------+--------------+



Q8. Identify the top 3 highest earners (Annual Salary) within each Department

In [ ]:
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window as W

windowSpec = W.partitionBy('Department').orderBy(desc('Annual Salary'))
df.withColumn('rnk',dense_rank().over(windowSpec)) \
.filter('rnk <= 3') \
.drop('rnk') \
.select('First Name','Last Name','Department','Annual Salary') \
.show(truncate=False)

+-----------+-----------+---------------------------+-------------+
|First Name |Last Name  |Department                 |Annual Salary|
+-----------+-----------+---------------------------+-------------+
|Ashraf     |Aleid      |Account Management         |41400        |
|Sana       |Alhusayn   |Account Management         |40944        |
|Muhamad    |Alaya      |Account Management         |40728        |
|Khawla     |Dawi       |Creative                   |40656        |
|Tariq      |Zaeur      |Creative                   |30528        |
|Fatin      |Red        |Creative                   |29052        |
|Ahmad      |Dabana     |Environmental Compliance   |40248        |
|Ghalib     |Zakianiin  |Environmental Compliance   |37812        |
|Safa       |AlAhmar    |Environmental Compliance   |37296        |
|Ayat       |Byd        |Environmental Health/Safety|37440        |
|Ghaliah    |Eibad      |Environmental Health/Safety|31020        |
|Muhamad    |Ayman      |Environmental Health/Sa

Q9. Filter and display all employees who earn a Monthly Salary greater than the average salary of their respective Department

In [ ]:
from pyspark.sql.functions import round,avg
from pyspark.sql.window import Window as W

windowSpec = W.partitionBy('Department')
df.withColumn('Dept_Avg_Salary',round(avg('Monthly Salary').over(windowSpec),2)) \
.where('`Monthly Salary` > `Dept_Avg_Salary`') \
.orderBy('Id') \
.select('Id','First Name','Last Name','Monthly Salary','Dept_Avg_Salary') \
.show(truncate=False)

+---+-----------+-----------+--------------+---------------+
|Id |First Name |Last Name  |Monthly Salary|Dept_Avg_Salary|
+---+-----------+-----------+--------------+---------------+
|2  |Omar       |Hishan     |3247          |2053.96        |
|3  |Ailya      |Sharaf     |2506          |2242.38        |
|6  |Muhamad    |Zueitr     |2332          |1964.71        |
|7  |Iin        |Alhalaliu  |1959          |1956.45        |
|8  |Muhamad    |Alaya      |3394          |1937.17        |
|14 |Iilian     |Dbs        |3404          |2285.28        |
|17 |Sandra     |Aljurmqaniu|3149          |1937.17        |
|19 |Ahed       |Salim      |2162          |2053.96        |
|20 |Ayham      |Tutwnji    |2180          |1964.71        |
|24 |Ahmad      |Ahmad      |2682          |2083.93        |
|25 |Muhamad    |Altarah    |3044          |2114.53        |
|27 |Jalal      |Almuluhi   |2207          |2053.96        |
|28 |Rana       |Mius       |2136          |2004.64        |
|31 |Eala       |Alhaj  

Q10. Create a new column called Cumulative_Salary that calculates the running total of Monthly Salary within each Department, ordered by their Start Date

In [ ]:
from pyspark.sql.functions import round,avg
from pyspark.sql.window import Window as W

windowSpec = W.partitionBy('Department').orderBy('Start Date')
result = df.withColumn('Start Date', to_date(df['Start Date'], 'dd-MM-yyyy'))
result.withColumn('Cumulative_Salary',sum('Monthly Salary').over(windowSpec)) \
.select('First Name','Last Name','Start Date','Department','Monthly Salary','Cumulative_Salary') \
.show(100)

+------------+-----------+----------+------------------+--------------+-----------------+
|  First Name|  Last Name|Start Date|        Department|Monthly Salary|Cumulative_Salary|
+------------+-----------+----------+------------------+--------------+-----------------+
|        Mari|       Ayly|2016-04-20|Account Management|          1639|             1639|
|       Husam|     Alsaed|2016-05-12|Account Management|          2147|             3786|
|      Sandra|Aljurmqaniu|2016-05-27|Account Management|          3149|             6935|
|       Amira|      Akrym|2016-06-23|Account Management|           898|             7833|
|      Khalid|   Alkhawam|2016-07-06|Account Management|          3157|            10990|
|       Mazin|     Earabi|2016-09-13|Account Management|          1171|            12161|
|       Shadi|      Qitan|2017-04-04|Account Management|          2613|            14774|
|       Hasan|    Aljasim|2017-05-24|Account Management|          2082|            16856|
|       Sa

Q11. Create a new column named Leave_Utilization.

If an employee has 0 Sick Leaves and 0 Unpaid Leaves, set the value to 'Perfect Attendance'.

If the sum of their Sick Leaves and Unpaid Leaves is greater than 5, set it to 'High Leave Usage'.

For everyone else, set it to 'Standard'.

In [ ]:
from pyspark.sql.functions import when

df.withColumn('Leave_Utilization',
              when((df["Sick Leaves"] == 0) & (df["Unpaid Leaves"] == 0),"Perfect Attendance") \
              .when((df['Sick Leaves'] + df['Unpaid Leaves'] > 5),"High Leave Usage") \
              .otherwise('Standard') \
              ).select('First Name','Last Name','Sick Leaves','Unpaid Leaves','Leave_Utilization') \
              .show(truncate=False)

+----------+-----------+-----------+-------------+------------------+
|First Name|Last Name  |Sick Leaves|Unpaid Leaves|Leave_Utilization |
+----------+-----------+-----------+-------------+------------------+
|Ghadir    |Hmshw      |1          |0            |Standard          |
|Omar      |Hishan     |0          |5            |Standard          |
|Ailya     |Sharaf     |0          |3            |Standard          |
|Lwiy      |Qbany      |0          |0            |Perfect Attendance|
|Ahmad     |Bikri      |0          |5            |Standard          |
|Muhamad   |Zueitr     |3          |0            |Standard          |
|Iin       |Alhalaliu  |6          |0            |High Leave Usage  |
|Muhamad   |Alaya      |0          |0            |Perfect Attendance|
|Susin     |Almilat    |0          |0            |Perfect Attendance|
|Muhamad   |Alrifaei   |1          |0            |Standard          |
|Muhamad   |Alqadah    |5          |0            |Standard          |
|Muhamad   |Iad     

Q12. Find departments having more than 500 hours of Overtime Hours

In [ ]:
df.groupBy('Department').agg(sum('Overtime Hours').alias('Dept_Overtime_Hrs')).filter('Dept_Overtime_Hrs > 500').orderBy(desc('Dept_Overtime_Hrs')).show(truncate=False)

+----------------------+-----------------+
|Department            |Dept_Overtime_Hrs|
+----------------------+-----------------+
|Quality Control       |1700             |
|Manufacturing         |1699             |
|Account Management    |1264             |
|Quality Assurance     |771              |
|Facilities/Engineering|605              |
|IT                    |523              |
+----------------------+-----------------+



Q13. The Start Date column is currently a string formatted as dd-mm-yyyy. Convert this column into a proper Spark DateType and filter for employees who joined after January 1st, 2019.

In [ ]:
from pyspark.sql.functions import to_date

result = df.withColumn('Start Date', to_date(df['Start Date'], 'dd-MM-yyyy'))
result.filter(result['Start Date'] > '2019-01-01').show()

+---+----------+---------------+------+----------+--------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+
| Id|First Name|      Last Name|Gender|Start Date|          Department|             Country|Center|Monthly Salary|Annual Salary|Sick Leaves|Unpaid Leaves|Overtime Hours|
+---+----------+---------------+------+----------+--------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+
|  2|      Omar|         Hishan|  Male|2020-05-21|     Quality Control|        Saudi Arabia|  West|          3247|        38964|          0|            5|           198|
|  5|     Ahmad|          Bikri|  Male|2020-03-11|       Manufacturing|               Egypt|  Main|           970|        11640|          0|            5|           121|
|  7|       Iin|      Alhalaliu|Female|2020-05-08|               Sales|United Arab Emirates|  Main|          1959|        23508|          6|          

Q14. For each Country, order the employees by their Monthly Salary from lowest to highest. Create a column called Salary_Gap that calculates the exact difference in Monthly Salary between the current employee and the employee directly below them in earnings.

In [ ]:
from pyspark.sql.functions import lead

windowSpec = W.partitionBy('Country').orderBy('Monthly Salary')
result = df.withColumn('Next_Salary',lead('Monthly Salary').over(windowSpec))
result.withColumn('Salary_Gap',result['Next_Salary'] - result['Monthly Salary']).orderBy(desc('Salary_Gap')).show(truncate=False)

+---+----------+---------+------+----------+---------------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+-----------+----------+
|Id |First Name|Last Name|Gender|Start Date|Department                 |Country             |Center|Monthly Salary|Annual Salary|Sick Leaves|Unpaid Leaves|Overtime Hours|Next_Salary|Salary_Gap|
+---+----------+---------+------+----------+---------------------------+--------------------+------+--------------+-------------+-----------+-------------+--------------+-----------+----------+
|165|Abdalmjid |Ahmad    |Male  |22-06-2019|Manufacturing              |Lebanon             |Main  |1578          |18936        |0          |0            |15            |2068       |490       |
|179|Aishah    |Alquatliu|Female|07-03-2020|Quality Assurance          |Lebanon             |South |2068          |24816        |0          |1            |0             |2537       |469       |
|469|Muhamad   |Basil    |Male

Q15. Filter the dataset to only include employees who are from IT department. For this filtered group, calculate a 5% bonus on their Annual Salary and display the final output showing their First Name,Last Name Department, and the calculated Bonus_Amount sorted from highest to lowest.

In [ ]:
df.filter(df['Department'] == 'IT') \
.withColumn('Bonus_Amount',round(df['Annual Salary'] * 0.05,2)) \
.select('First Name','Last Name','Department','Bonus_Amount') \
.orderBy(desc('Bonus_Amount')) \
.show()

+-----------+---------------+----------+------------+
| First Name|      Last Name|Department|Bonus_Amount|
+-----------+---------------+----------+------------+
|    Labanaa|        Qaziyha|        IT|      1954.8|
|    Muhamad|Sharif Aldaghly|        IT|      1954.2|
|    Muhamad|          Khayr|        IT|      1946.4|
|    Muhamad|         Khalid|        IT|      1895.4|
|      Darin|         Ghunum|        IT|      1894.2|
|    Muhamad|        Altarah|        IT|      1826.4|
|       Alaa|       Iismaeil|        IT|      1825.8|
|       Alaa|        Alyusif|        IT|      1811.4|
|        Asd|        Bwfaeur|        IT|      1757.4|
|     Muwmin|       Almudhin|        IT|      1729.2|
|      Akram|     Alhulwanii|        IT|      1674.0|
|    Kamilia|           Sydu|        IT|      1570.2|
|Abd Albasit|        AlAhmar|        IT|      1563.6|
|        Iin|        Farahat|        IT|      1487.4|
|      Lilas|        Balatah|        IT|      1440.6|
|      Ahmad|        Alealbi